# 02 CCPD to YOLO: visual check (M1b-2)
Runs on Colab (Drive mounted). Builds `ccpd_yolo.tar` with `python -m alpr.ccpd yolo`, extracts it on the VM, checks counts and draws the converted boxes on a seeded sample.
**This notebook shows CCPD images** (the CCPD repo terms are unchecked, see PLAN.md). The repo is public: clear all outputs before committing (see the last cell).

In [ ]:
import os, random, shutil, tarfile
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw

In [ ]:
# Parameters
SEED = 0
DRIVE = '/content/drive/MyDrive/alpr'
ARCHIVES = f'{DRIVE}/archives'
SAMPLE_TAR = f'{ARCHIVES}/ccpd_sample.tar'  # input, built in M1a
SPLIT_CSV = f'{ARCHIVES}/ccpd_split.csv'    # input, M1b-1
YOLO_TAR = f'{ARCHIVES}/ccpd_yolo.tar'      # output, kept on Drive
YOLO_TAR_LOCAL = '/content/ccpd_yolo.tar'   # VM copy
YOLO_DIR = '/content/ccpd_yolo'             # extracted on the VM
REPORTS = f'{DRIVE}/reports/ccpd_yolo'      # the grid PNG goes here
N_SAMPLE = 6   # random images per split in the grid
N_EXTREME = 4  # smallest and largest boxes (keep <= N_SAMPLE)

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition

In [ ]:
from alpr import ccpd  # ImportError on the first run? Restart the kernel once: an editable install is not visible to an already-running kernel.
print('alpr.ccpd:', ccpd.__file__)
for p in (SAMPLE_TAR, SPLIT_CSV):
    assert os.path.exists(p), f'missing on Drive: {p}'

## Step 3: convert
Reads the sample tar and the split file, checks every image and writes `ccpd_yolo.tar` plus `ccpd_yolo.dropped.csv` to Drive. If the tar already exists the command prints "skipping" and no counts; the dropped-per-reason table below is read from the CSV, so it shows either way. To see the command's own printout again, delete the tar and the CSV first.

In [ ]:
!python -m alpr.ccpd yolo "{SAMPLE_TAR}" "{SPLIT_CSV}" "{YOLO_TAR}"
dropped = pd.read_csv(Path(YOLO_TAR).with_suffix('.dropped.csv'))
print('dropped per (reason, split):')
print(dropped.groupby(['reason', 'split']).size() if len(dropped) else 'nothing dropped')

## Step 4: copy to the VM disk and extract
Training reads from local disk, never from the Drive mount. The VM copy must have the same MD5 as the Drive file.

In [ ]:
if not os.path.exists(YOLO_TAR_LOCAL):
    shutil.copyfile(YOLO_TAR, YOLO_TAR_LOCAL)
md5 = ccpd.file_md5(YOLO_TAR_LOCAL)
assert md5 == ccpd.file_md5(YOLO_TAR), 'VM copy differs from the Drive file'
print('ccpd_yolo.tar MD5 (record in PLAN.md):', md5)
if not Path(YOLO_DIR, 'data.yaml').exists():
    with tarfile.open(YOLO_TAR_LOCAL) as t:
        t.extractall(YOLO_DIR, filter='data')
print('extracted to', YOLO_DIR)

## Step 5: data.yaml and counts
Expected = rows in `ccpd_split.csv` minus the dropped images, per split. The assert fails if the extracted files do not match.

In [ ]:
root = Path(YOLO_DIR)
print(root.joinpath('data.yaml').read_text())
split = pd.read_csv(SPLIT_CSV)
split['base'] = split['member'].str.rsplit('/', n=1).str[-1]
want = split.groupby('split').size().sub(dropped.groupby('split').size(), fill_value=0).astype(int)
rows, names = [], set()
for s in ('train', 'val', 'heldout'):
    imgs = sorted(root.joinpath('images', s).glob('*.jpg'))
    labs = sorted(root.joinpath('labels', s).glob('*.txt'))
    assert [p.stem for p in imgs] == [p.stem for p in labs], f'{s}: images and labels differ'
    names |= {p.name for p in imgs}
    rows.append((s, len(imgs), len(labs), int(want[s])))
table = pd.DataFrame(rows, columns=['split', 'images', 'labels', 'expected'])
print(table.to_string(index=False))
assert (table['images'] == table['expected']).all(), 'counts differ from split csv minus dropped'
bad = 0
for p in root.glob('labels/*/*.txt'):
    f = p.read_text().split()
    bad += not (len(f) == 5 and f[0] == '0' and all(0 < float(v) <= 1 for v in f[1:]))
print('label files not in the form `0 cx cy w h` within (0, 1]:', bad)
assert bad == 0
print('kept per label and split (from the extracted files):')
kept = split[split['base'].isin(names)]
print(pd.crosstab(kept['subset'], kept['split'], margins=True))

## Step 6: draw the converted boxes
Seeded random sample (train and held-out), then the smallest and largest boxes over all splits. Titles show no filenames or plate text.

In [ ]:
def read_box(img: Path) -> tuple[float, float, float, float]:
    _, cx, cy, w, h = root.joinpath('labels', img.parent.name, img.stem + '.txt').read_text().split()
    return float(cx), float(cy), float(w), float(h)

def draw(img: Path) -> Image.Image:
    cx, cy, w, h = read_box(img)
    im = Image.open(img).convert('RGB')
    W, H = im.size
    ImageDraw.Draw(im).rectangle([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H], outline='red', width=5)
    return im

by_split = {s: sorted(root.joinpath('images', s).glob('*.jpg')) for s in ('train', 'val', 'heldout')}
by_area = sorted(((read_box(p)[2] * read_box(p)[3], p) for ps in by_split.values() for p in ps), key=lambda t: t[0])
rng = random.Random(SEED)
panels = [
    [(p, 'train') for p in rng.sample(by_split['train'], N_SAMPLE)],
    [(p, 'held-out') for p in rng.sample(by_split['heldout'], N_SAMPLE)],
    [(p, f'smallest: {a:.2%} of image') for a, p in by_area[:N_EXTREME]],
    [(p, f'largest: {a:.2%} of image') for a, p in by_area[-N_EXTREME:]],
]
fig, axes = plt.subplots(len(panels), N_SAMPLE, figsize=(3.2 * N_SAMPLE, 2.6 * len(panels)))
for ax in axes.ravel():
    ax.axis('off')
for row, items in zip(axes, panels):
    for ax, (p, title) in zip(row, items):
        ax.imshow(draw(p))
        ax.set_title(title, fontsize=9)
fig.suptitle('CCPD converted YOLO boxes (red)')
fig.tight_layout()

## Step 7: save the grid to Drive
PNG goes to the private `reports/ccpd_yolo` folder on Drive, for the presentation. It must not be copied into the repo.

In [ ]:
os.makedirs(REPORTS, exist_ok=True)
out = f'{REPORTS}/ccpd_yolo_grid.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('saved', out)

## Before committing this notebook
This notebook is the exception to "keep executed outputs": its outputs contain CCPD images. After Restart & Run All, download it and, from the repo root, run
`uv run --with nbconvert jupyter nbconvert --clear-output --inplace notebooks/02_ccpd_yolo_check.ipynb`
then check `grep -c image/png notebooks/02_ccpd_yolo_check.ipynb` prints 0. Copy the counts from step 3 and 5 into PLAN.md by hand.